## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Applies a precomputed mean-difference steering vector to the residual stream during generation, alongside activation patching, and measures how it shifts the model's factual vs. counterfactual answer probability.

### Set-up

In [ ]:
import torch
import gc
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, prepare_batch_multitoken_steering

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_penultimate_sum", # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
)
intervention_config = _config.InterventionConfig(
    intervention_loc="", # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
)
run_config = _config.RunConfig(
    experiment_root="experiments/steering",
    result_dir="steering",
    overwrite=True,
)
suffix = "_intervention_effect_var20"

# Steering Config———————————————————————————————————————
batch_size = 24
device = "cuda"

In [4]:
num_layers = 24
hidden_size = 2880
model, tokenizer = _config.load_model(prompt_config.model_type)
if "GPT-OSS" in prompt_config.model_type:
    num_layers = model.config.num_hidden_layers
    hidden_size = model.config.hidden_size

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [6]:
intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
print(intervention_ids)

[20]


In [7]:
tok_pos_list = _config.build_tok_pos_list(intervention_config.tok_pos_fn, intervention_ids)
print(tok_pos_list)

[235]


## Intervening

Loads the steering vector for a chosen layer and direction, applies it together with an activation patch during generation, and logs the resulting factual vs. counterfactual answer probabilities.

In [24]:
STEERING_LAYER = 6
STEERING_DIRECTION = "positive"
steering_vectors = torch.load("experiments/steering/output/GPT-OSS_stepwise/intervention_effect_tensors/h_pre_penultimate_sum/intervention_effect.pt", weights_only=False, map_location=device)
steering_vector = steering_vectors[str(STEERING_LAYER)]
# steering_vector = torch.load("experiments/steering/steering_vectors_gpt-oss-20b.pt")["backtracking"]["mean"][STEERING_LAYER]

In [9]:
for layer_str, vec in steering_vectors.items():
    norm = torch.norm(vec).item()
    print(f"Layer {layer_str} norm: {norm}")

Layer 0 norm: 9.017988204956055
Layer 1 norm: 21.314414978027344
Layer 2 norm: 56.38667678833008
Layer 3 norm: 70.55854797363281
Layer 4 norm: 88.75227355957031
Layer 5 norm: 110.6183090209961
Layer 6 norm: 134.1468963623047
Layer 7 norm: 177.43605041503906
Layer 8 norm: 266.2468566894531
Layer 9 norm: 340.4071960449219
Layer 10 norm: 427.3614501953125
Layer 11 norm: 528.5487060546875
Layer 12 norm: 676.5377197265625
Layer 13 norm: 880.3982543945312
Layer 14 norm: 1148.875244140625
Layer 15 norm: 1589.6180419921875
Layer 16 norm: 2059.71337890625
Layer 17 norm: 2662.899169921875
Layer 18 norm: 3471.234130859375
Layer 19 norm: 4622.0537109375
Layer 20 norm: 6195.85107421875
Layer 21 norm: 7365.794921875
Layer 22 norm: 8937.013671875
Layer 23 norm: 10434.2861328125


In [25]:
coefficient = 1
if STEERING_DIRECTION == "negative":
    coefficient *= -1

run_config = _config.RunConfig(
    experiment_root="experiments/steering",
    result_dir="steering",
    output_filename=f"filtered_{prompt_config.stem}_{coefficient}{suffix}_l{STEERING_LAYER}.csv",
    overwrite=True,
)
header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]
    factual_labels, counterfactual_labels = _config.prepare_label_tensors(tokenizer, batch_rows)

    hooks = []
    tokens, counterfactual_tokens, hooks = prepare_batch_multitoken_intervention(model, tokenizer, [0], tok_pos_list, batch_rows['base_prompt'].tolist(), batch_rows['source_prompt'].tolist())
    hooks.extend(hooks)
    steering_tokens, steering_hook = prepare_batch_multitoken_steering(model, tokenizer, STEERING_LAYER, tok_pos_list, batch_rows['base_prompt'].tolist(), coefficient * steering_vector)
    hooks.append(steering_hook)

    assert torch.equal(tokens['input_ids'], steering_tokens['input_ids'])
        
    input_length = tokens["input_ids"].shape[1]

    with torch.no_grad():
        output = batch_intervene(model, tokens["input_ids"], hooks, attention_mask=tokens["attention_mask"]) # attention_freeze_hooks + 
    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
    counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
    del output
    
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(pred_toks[j].unsqueeze(-1))
        _config.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text, factual_prob[j].item(), counterfactual_prob[j].item()])

    del tokens, hooks
    torch.cuda.empty_cache()
    gc.collect()


100%|█████████████████████████████████████████████████████████████████████████████| 11/11 [01:43<00:00,  9.43s/it]


In [11]:
print(steering_vector)
print(torch.norm(steering_vector))

tensor([-0.2854, -5.0718, -1.4510,  ...,  0.0000,  0.0000,  0.0000],
       device='cuda:7')
tensor(41.1911, device='cuda:7')
